***DAY 3***

My goal today is to first refine the projection system to be more accurate for young players, and then to start work on a market value system that will be used going forward

My main issue with the valuation so far is that young players (ie 2+ yrs experience) are being projected to regress WAY too much. While it is reasonable to expect regression to the mean from mid career players, a big year of development for a 21 year old should also be seen as a positive. The real goal will be to make it so that young developing players are expected to continue development, but also so that regression to the mean is still accounted for, and there isn't a positive feedback loop surrounding growth projections

Lets start working on a delta_ps x age interaction variable

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [6]:
df = pd.read_csv('player_stats.csv')
df['delta_ps'] = df['delta_ps'].round(1)
df = df[df['season'] != '2025-26']

fill_cols = ['point_shares_prev1', 'point_shares_prev2', 'point_shares_prev3', 'delta_ps']
df[fill_cols] = df[fill_cols].fillna(0)
df = df.dropna(subset=['age'])
df = df.dropna(subset=['next_season_ps'])

In [7]:
# Cumulative GP residual
slope, intercept, r, p, se = stats.linregress(df['age'], df['cumulative_gp'])
df['cumulative_gp_residual'] = df['cumulative_gp'] - (slope * df['age'] + intercept)

# Draft round weighted
df['draft_round_weighted'] = df.apply(
    lambda row: row['draft_round'] if 20 <= row['age'] <= 25 else 0, axis=1
)

# Delta PS x age interaction
df['delta_ps_age_interaction'] = df['delta_ps'] * df['age']

In [8]:
features_clean = [
    'point_shares', 'point_shares_prev1', 'point_shares_prev2', 'point_shares_prev3',
    'games_played', 'cumulative_gp_residual', 'draft_round_weighted',
    'delta_ps', 'age', 'delta_ps_age_interaction'
]

X = df[features_clean]
y = df['next_season_ps']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

final_model = LinearRegression()
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
print(f"R²: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {mean_squared_error(y_test, y_pred)**0.5:.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")

coef_df = pd.DataFrame({'feature': features_clean, 'coefficient': final_model.coef_}).sort_values('coefficient', ascending=False)
print(coef_df)

R²: 0.5264
RMSE: 1.9891
MAE: 1.4835
                    feature  coefficient
0              point_shares     0.733114
2        point_shares_prev2     0.112303
3        point_shares_prev3     0.069645
4              games_played     0.002352
5    cumulative_gp_residual     0.000207
7                  delta_ps    -0.008680
9  delta_ps_age_interaction    -0.010146
6      draft_round_weighted    -0.027355
1        point_shares_prev1    -0.055158
8                       age    -0.137811


Well that is not what we wanted. The R^2 went down, signifying that we are getting worse results with the added feature. lets look at what happened

In [9]:
df['age_group'] = pd.cut(df['age'], bins=[18, 21, 24, 27, 30, 45], labels=['18-21', '22-24', '25-27', '28-30', '30+'])
df['delta_ps_positive'] = df['delta_ps'] > 0

summary = df.groupby(['age_group', 'delta_ps_positive'])['next_season_ps'].mean().reset_index()
print(summary)

  age_group  delta_ps_positive  next_season_ps
0     18-21              False        1.902042
1     18-21               True        3.403888
2     22-24              False        2.050121
3     22-24               True        3.123039
4     25-27              False        2.700916
5     25-27               True        3.159166
6     28-30              False        2.813199
7     28-30               True        3.362597
8       30+              False        2.830082
9       30+               True        2.979039


This shows some of my intuition is right. Young players who see an increase in PS will be better than those that don,t and the difference between positive and negative delta_ps in terms of what next season outlook should be is way bigger than with old players. It may help to flip the way age is handled?

In [10]:
df['delta_ps_age_interaction'] = df['delta_ps'] * (1 / df['age'])

features_clean = [
    'point_shares', 'point_shares_prev1', 'point_shares_prev2', 'point_shares_prev3',
    'games_played', 'cumulative_gp_residual', 'draft_round_weighted',
    'delta_ps', 'age', 'delta_ps_age_interaction'
]

X = df[features_clean]
y = df['next_season_ps']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

final_model = LinearRegression()
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
print(f"R²: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {mean_squared_error(y_test, y_pred)**0.5:.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")

coef_df = pd.DataFrame({'feature': features_clean, 'coefficient': final_model.coef_}).sort_values('coefficient', ascending=False)
print(coef_df)

R²: 0.5265
RMSE: 1.9889
MAE: 1.4834
                    feature  coefficient
9  delta_ps_age_interaction     7.652016
0              point_shares     0.734392
2        point_shares_prev2     0.112514
3        point_shares_prev3     0.069817
4              games_played     0.002305
5    cumulative_gp_residual     0.000205
6      draft_round_weighted    -0.026933
1        point_shares_prev1    -0.056699
8                       age    -0.137065
7                  delta_ps    -0.573775


Still worse than our original model. Lets take another approach: only have the variable work for young players (under 23).

In [12]:
df['delta_ps_young'] = df.apply(
    lambda row: row['delta_ps'] if row['age'] < 23 else 0, axis=1
)

features_clean = [
    'point_shares', 'point_shares_prev1', 'point_shares_prev2', 'point_shares_prev3',
    'games_played', 'cumulative_gp_residual', 'draft_round_weighted',
    'delta_ps', 'age', 'delta_ps_young'
]

X = df[features_clean]
y = df['next_season_ps']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

final_model = LinearRegression()
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
print(f"R²: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {mean_squared_error(y_test, y_pred)**0.5:.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")

coef_df = pd.DataFrame({'feature': features_clean, 'coefficient': final_model.coef_}).sort_values('coefficient', ascending=False)
print(coef_df)

R²: 0.5263
RMSE: 1.9893
MAE: 1.4831
                  feature  coefficient
0            point_shares     0.733555
2      point_shares_prev2     0.110317
9          delta_ps_young     0.089095
3      point_shares_prev3     0.070758
4            games_played     0.002391
5  cumulative_gp_residual     0.000212
6    draft_round_weighted    -0.025488
1      point_shares_prev1    -0.054349
8                     age    -0.136012
7                delta_ps    -0.293786


Honestly at this point it looks like we might just have to accept that this feature won't be involved in the model. Doing so doesn't seem to really help all that much. However, when we calculate market values for players, we are going to be doing so with financial methods that will allow us to account for age and growth in their long term valuation.

**part 2**

Okay we are actually gonna hold off on doing market value research just yet. We can equate picks to cap relief based off of work that eric tulsky published about that, but I'll need to scrape some data on players signing contracts to make the model. Right now we are gonna try and calculate the total cap hit that a player should earn over x no. of years due to their point shares and expected change in point shares over the years. 

To do this we are gonna want to predict delta_ps as a function of age and current_ps. Realistically we can evalue a players expected value curve over the next few years as a result of this.

As we have already seen with the model so far, young players development is not handled perfectly, but we want to make sure that we are predicting players to develop accurately when they are young when we are coming up with contract values. As a result we are going to make a seperate delta_ps regression for each age. Then I will plug it into the excel book.

In [17]:
aging_models = {}

for age in range(18, 34):
    age_df = df[df['age'] == age] if age < 33 else df[df['age'] >= 33]
    
    if len(age_df) < 30:  # skip if too few samples
        print(f"Age {age}: insufficient data ({len(age_df)} rows), skipping")
        continue
    
    X_age = age_df[['point_shares', 'draft_round']]
    y_age = age_df['delta_ps']
    
    model_age = LinearRegression()
    model_age.fit(X_age, y_age)
    
    r2 = r2_score(y_age, model_age.predict(X_age))
    print(f"Age {age}: n={len(age_df)}, R²={r2:.4f}, "
          f"ps_coef={model_age.coef_[0]:.4f}, "
          f"draft_coef={model_age.coef_[1]:.4f}, "
          f"intercept={model_age.intercept_:.4f}")
    
    aging_models[age] = model_age

Age 18: n=110, R²=1.0000, ps_coef=0.0000, draft_coef=0.0000, intercept=0.0000
Age 19: n=411, R²=0.1524, ps_coef=0.1979, draft_coef=0.0310, intercept=-0.2000
Age 20: n=981, R²=0.2537, ps_coef=0.3197, draft_coef=0.0393, intercept=-0.2579
Age 21: n=1439, R²=0.3042, ps_coef=0.3825, draft_coef=0.0490, intercept=-0.3167
Age 22: n=1721, R²=0.2349, ps_coef=0.3650, draft_coef=0.1026, intercept=-0.6440
Age 23: n=1853, R²=0.2077, ps_coef=0.3341, draft_coef=0.0844, intercept=-0.5993
Age 24: n=1758, R²=0.2343, ps_coef=0.3496, draft_coef=0.1280, intercept=-1.0358
Age 25: n=1612, R²=0.1405, ps_coef=0.2828, draft_coef=0.1024, intercept=-1.0589
Age 26: n=1422, R²=0.1723, ps_coef=0.3152, draft_coef=0.1402, intercept=-1.4337
Age 27: n=1250, R²=0.1716, ps_coef=0.3297, draft_coef=0.1391, intercept=-1.4705
Age 28: n=1103, R²=0.1735, ps_coef=0.3353, draft_coef=0.0994, intercept=-1.5540
Age 29: n=937, R²=0.1679, ps_coef=0.3659, draft_coef=0.0693, intercept=-1.7553
Age 30: n=792, R²=0.2120, ps_coef=0.3804, dra

Okay! R^2 isn'y super high but at the end of the day that is okay. I also think that draft_coef is mostly noise at this point and should be dropped. You might notice that Age 33+ players are all lumped together. The goal is to just have a larger sample size for old, regressing players. I'll now throw this in the workbook and create a way to predict what player development will look like going forward.

In [18]:
aging_models = {}

for age in range(18, 34):
    age_df = df[df['age'] == age] if age < 33 else df[df['age'] >= 33]
    
    if len(age_df) < 30:
        print(f"Age {age}: insufficient data ({len(age_df)} rows), skipping")
        continue
    
    X_age = age_df[['point_shares']]
    y_age = age_df['delta_ps']
    
    model_age = LinearRegression()
    model_age.fit(X_age, y_age)
    
    r2 = r2_score(y_age, model_age.predict(X_age))
    print(f"Age {age}: n={len(age_df)}, R²={r2:.4f}, "
          f"ps_coef={model_age.coef_[0]:.4f}, "
          f"intercept={model_age.intercept_:.4f}")
    
    aging_models[age] = model_age

Age 18: n=110, R²=1.0000, ps_coef=0.0000, intercept=0.0000
Age 19: n=411, R²=0.1518, ps_coef=0.1953, intercept=-0.1535
Age 20: n=981, R²=0.2519, ps_coef=0.3123, intercept=-0.1669
Age 21: n=1439, R²=0.3014, ps_coef=0.3716, intercept=-0.1784
Age 22: n=1721, R²=0.2228, ps_coef=0.3423, intercept=-0.3166
Age 23: n=1853, R²=0.1988, ps_coef=0.3151, intercept=-0.3015
Age 24: n=1758, R²=0.2148, ps_coef=0.3232, intercept=-0.5565
Age 25: n=1612, R²=0.1283, ps_coef=0.2593, intercept=-0.6552
Age 26: n=1422, R²=0.1506, ps_coef=0.2881, intercept=-0.8880
Age 27: n=1250, R²=0.1515, ps_coef=0.3056, intercept=-0.9319
Age 28: n=1103, R²=0.1631, ps_coef=0.3190, intercept=-1.1678
Age 29: n=937, R²=0.1635, ps_coef=0.3555, intercept=-1.4940
Age 30: n=792, R²=0.2043, ps_coef=0.3697, intercept=-1.4070
Age 31: n=641, R²=0.1412, ps_coef=0.3309, intercept=-1.4473
Age 32: n=513, R²=0.1344, ps_coef=0.3244, intercept=-1.5603
Age 33: n=1167, R²=0.1915, ps_coef=0.4096, intercept=-1.9505


Okay so one issue that is now emerging to me with using this method is that the data predicts any player with a point shares value over 5 to just continue regressing pretty much at any age. We should have a cutoff where we stop predicting progression altogether. Lets do this by selecting the age at which only x% of players see a positive delta_ps

In [19]:
age_positive_delta = df.groupby('age').apply(
    lambda x: (x['delta_ps'] > 0).mean() * 100
).reset_index()
age_positive_delta.columns = ['age', 'pct_positive_delta']
age_positive_delta = age_positive_delta[(age_positive_delta['age'] >= 18) & (age_positive_delta['age'] <= 40)]
print(age_positive_delta.to_string())

     age  pct_positive_delta
0   18.0            0.000000
1   19.0           13.868613
2   20.0           24.566769
3   21.0           36.483669
4   22.0           42.068565
5   23.0           47.436589
6   24.0           47.326507
7   25.0           46.277916
8   26.0           47.116737
9   27.0           46.000000
10  28.0           46.690843
11  29.0           43.756670
12  30.0           46.590909
13  31.0           45.241810
14  32.0           41.325536
15  33.0           44.730077
16  34.0           37.704918
17  35.0           39.719626
18  36.0           44.715447
19  37.0           37.142857
20  38.0           30.303030
21  39.0           47.058824
22  40.0            0.000000


Okay honestly it looks like just a lot of noise here. I am just gonna make the call to have the cutoff at 31. I know that it is unscientific but the common knowledge is that most players end their peak around age 30 or 31. After that, we will just use an age-only regression that will guarantee regression for all players who have left their prime. While it may not be based on the most thorough analytical backing, it follows with how a team would probably want to project players' trajectories.

In [21]:
aging_models_decline = {}

for age in range(30, 34):
    age_df = df[df['age'] == age] if age < 33 else df[df['age'] >= 33]
    
    if len(age_df) < 30:
        print(f"Age {age}: insufficient data ({len(age_df)} rows), skipping")
        continue
    
    # Just intercept - no features, just average delta_ps at that age
    avg_delta = age_df['delta_ps'].mean()
    print(f"Age {age}: n={len(age_df)}, avg_delta_ps={avg_delta:.4f}")
    
    aging_models_decline[age] = avg_delta

Age 30: n=792, avg_delta_ps=-0.1453
Age 31: n=641, avg_delta_ps=-0.2883
Age 32: n=513, avg_delta_ps=-0.4524
Age 33: n=1167, avg_delta_ps=-0.4908


Okay one more issue with my thing is that there is no compressing factor for top players. For example Connor Bedard is projected to get 45 point shares at 30. This is  unrealistic.

Okay i included a dampening factor that starts with any player who is getting an expected point share higher than 6. It is linearly dampened up until 15 point shares, in which case a player will receive no growth. Now projections look a lot more reasonable

*Important note*

This aspect of the model is definitely a bit rough and as such is not meant as a projection of a players growth, but as a way to evaluate them as a sort of annuity with growth over time.

**pricing**

We are going to assign prices based on average point shares over the life of the contract, and by percentage of cap hit. To do so I got the average point shares for each time frame that a contract could be for a player, and weigthed that by the amount that the cap is going to grow over the next years. Since the 2012-13 negotiations, the cap has grown 4.75% per year on average, but a lot of that is weighed down by the flat cap of the COVID era. As a result, I estimated 5.5% growth going forward on average.